# DeepMzyme Colab Training Notebook

This notebook runs DeepMzyme training from Google Colab using the existing command-line interface in `src/train.py`. It is designed for baseline-first experiments on the trusted non-overlapped PinMyMetal split, with clear controls for task, model preset, hyperparameters, output copying, and run summarization.

The notebook does not duplicate training logic. It builds a CLI command, runs it only when you explicitly enable training, then calls `src/report_runs.py` to create a comparison CSV and optional figure.

## 1. Mount Google Drive

Use Drive for the compressed Colab bundle and for long-term run outputs. Runtime-local paths under `/content` are faster but disappear when the Colab session ends.

In [ ]:
#@title Mount Google Drive
MOUNT_DRIVE = True  #@param {type:"boolean"}

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Drive mount skipped. Make sure bundle and output paths are accessible.')

## 2. Configure Repository, Bundle, and Output Paths

Edit this cell first. The notebook always clones DeepMzyme from Git into `REPO_DIR`, so set `REPO_GIT_URL` and `REPO_GIT_BRANCH` before running the dependency cell.

`DRIVE_DATA_DIR` is the persistent Google Drive `DeepMzyme_Data` directory. The notebook reads the compressed bundle from `DRIVE_DATA_DIR/DeepMzyme_Colab_Bundles`, unpacks it into runtime-local `/content`, trains into runtime-local `/content`, then copies final run outputs back to `DRIVE_DATA_DIR/Train_Parameters_Models_and_Results`.

In [ ]:
#@title Paths and repository setup
from pathlib import Path

REPO_DIR = '/content/DeepMzyme'  #@param {type:"string"}
REPO_GIT_URL = 'https://github.com/MECHTI1/DeepMzyme.git'  #@param {type:"string"}
REPO_GIT_BRANCH = 'main'  #@param {type:"string"}

DRIVE_DATA_DIR = '/content/drive/MyDrive/DeepMzyme/DeepMzyme_Data'  #@param {type:"string"}
BUNDLE_FILENAME = 'train_and_test_sets_structures_non_overlapped_pinmymetal_colab_bundle_structures.tar.zst'  #@param {type:"string"}
UNPACK_DIR = '/content/deepmzyme_bundle'  #@param {type:"string"}
DATASET_NAME = 'train_and_test_sets_structures_non_overlapped_pinmymetal'  #@param {type:"string"}

LOCAL_RUNS_DIR = '/content/deepmzyme_runs'  #@param {type:"string"}
DRIVE_OUTPUT_SUBDIR = 'Train_Parameters_Models_and_Results'  #@param {type:"string"}

repo_dir = Path(REPO_DIR).expanduser().resolve()
drive_data_dir = Path(DRIVE_DATA_DIR).expanduser()
BUNDLE_PATH = str(drive_data_dir / 'DeepMzyme_Colab_Bundles' / BUNDLE_FILENAME)
bundle_path = Path(BUNDLE_PATH).expanduser()
unpack_dir = Path(UNPACK_DIR).expanduser().resolve()
local_runs_dir = Path(LOCAL_RUNS_DIR).expanduser().resolve()
drive_output_dir = drive_data_dir / DRIVE_OUTPUT_SUBDIR

print('Repository:', repo_dir)
print('Drive DeepMzyme_Data directory:', drive_data_dir)
print('Bundle:', bundle_path)
print('Unpack directory:', unpack_dir)
print('Local runs directory:', local_runs_dir)
print('Drive output directory:', drive_output_dir)

## 3. Install and Check Dependencies

This cell prepares the repository and checks core imports. Installing dependencies can take several minutes in a fresh Colab runtime. If you already prepared the runtime, set `INSTALL_REQUIREMENTS` to `False` and keep the import check enabled.

In [ ]:
#@title Repository and dependency checks
import os
import subprocess
import sys
from pathlib import Path

INSTALL_REQUIREMENTS = True  #@param {type:"boolean"}
CHECK_IMPORTS = True  #@param {type:"boolean"}

if not REPO_GIT_URL or '<' in REPO_GIT_URL:
    raise ValueError('Set REPO_GIT_URL to your DeepMzyme repository URL before cloning.')
if repo_dir.exists():
    print(f'Repository already exists at {repo_dir}; skipping clone.')
else:
    subprocess.run(['git', 'clone', '--branch', REPO_GIT_BRANCH, REPO_GIT_URL, str(repo_dir)], check=True)

requirements_path = repo_dir / 'src' / 'requirements.txt'
if INSTALL_REQUIREMENTS:
    if requirements_path.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)], check=True)
    else:
        subprocess.run([
            sys.executable, '-m', 'pip', 'install',
            'torch-geometric==2.7.0', 'biopython==1.87', 'biotite>=1.6', 'propka>=3.5', 'gemmi>=0.7', 'pandas', 'matplotlib'
        ], check=True)

if CHECK_IMPORTS:
    src_path = str(repo_dir / 'src')
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    import torch
    import pandas as pd
    import torch_geometric
    print('Python:', sys.executable)
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    print('PyTorch Geometric:', torch_geometric.__version__)

## 4. Unpack the Colab Bundle

This extracts the training/test structures and CSV files into `UNPACK_DIR`. The notebook keeps extracted data outside the repository `DeepMzyme_Data` tree so smoke or setup work does not modify project data.

In [ ]:
#@title Unpack bundle
import shutil
import subprocess
from pathlib import Path

FORCE_REUNPACK = False  #@param {type:"boolean"}

def run_checked(cmd):
    print('+', ' '.join(str(part) for part in cmd))
    subprocess.run([str(part) for part in cmd], check=True)

def unpack_archive(source: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    suffixes = ''.join(source.suffixes).lower()
    if suffixes.endswith('.tar.zst'):
        if shutil.which('zstd') is None:
            run_checked(['apt-get', 'update'])
            run_checked(['apt-get', 'install', '-y', 'zstd'])
        run_checked(['tar', '--use-compress-program=zstd', '-xf', source, '-C', destination])
    elif suffixes.endswith(('.tar.gz', '.tgz', '.tar')):
        run_checked(['tar', '-xf', source, '-C', destination])
    else:
        raise ValueError(f'Unsupported bundle archive type: {source}')

if FORCE_REUNPACK and unpack_dir.exists():
    shutil.rmtree(unpack_dir)

if bundle_path.is_dir():
    unpack_dir = bundle_path.resolve()
    print(f'Using already-unpacked bundle directory: {unpack_dir}')
else:
    if not bundle_path.exists():
        raise FileNotFoundError(f'Bundle not found: {bundle_path}')
    marker = unpack_dir / '.deepmzyme_bundle_unpacked'
    if marker.exists() and not FORCE_REUNPACK:
        print(f'Bundle already unpacked at {unpack_dir}. Set FORCE_REUNPACK=True to extract again.')
    else:
        unpack_archive(bundle_path, unpack_dir)
        marker.write_text('ok\n', encoding='utf-8')
        print(f'Unpacked bundle into {unpack_dir}')

## 5. Verify Train/Test Directories and Summary CSVs

DeepMzyme training needs a train structure directory, a test structure directory, and MAHOMES-style summary CSVs with site-level columns such as PDB ID, EC number, metal residue number, and metal residue type. The bundle may also contain structure-level CSV artifacts; those are useful metadata, but they are not used here unless they match the training loader requirements.

In [ ]:
#@title Detect and verify dataset paths
import csv
from pathlib import Path

TRAIN_DIR_OVERRIDE = ''  #@param {type:"string"}
TEST_DIR_OVERRIDE = ''  #@param {type:"string"}
TRAIN_CSV_OVERRIDE = ''  #@param {type:"string"}
TEST_CSV_OVERRIDE = ''  #@param {type:"string"}

REQUIRED_SUMMARY_ALIASES = {
    'pdbid': {'pdbid', 'structure'},
    'metal residue number': {'metal residue number', 'chain_resi'},
    'EC number': {'ec number', 'ecnumber'},
    'metal residue type': {'metal residue type', 'metaltype'},
}

def has_required_summary_columns(path: Path) -> bool:
    try:
        with path.open('r', encoding='utf-8', newline='') as handle:
            reader = csv.DictReader(handle)
            fields = {field.strip().lower() for field in (reader.fieldnames or []) if field}
    except Exception:
        return False
    return all(fields.intersection(aliases) for aliases in REQUIRED_SUMMARY_ALIASES.values())

def find_dataset_root(root: Path, dataset_name: str) -> Path:
    candidates = [
        root / 'DeepMzyme_Data' / dataset_name,
        root / dataset_name,
        root,
    ]
    candidates.extend(path for path in root.rglob(dataset_name) if path.is_dir())
    for candidate in candidates:
        if (candidate / 'train').is_dir() and (candidate / 'test').is_dir():
            return candidate.resolve()
    raise FileNotFoundError(f'Could not find train/test directories for dataset {dataset_name!r} under {root}')

def find_summary_csv(split_dir: Path, split_name: str) -> Path:
    preferred_names = [
        'catalytic_only_summary.csv',
        'final_data_summarazing_table_transition_metals_only_catalytic.csv',
        f'{DATASET_NAME}_{split_name}.csv',
    ]
    candidates = [split_dir / name for name in preferred_names]
    candidates.extend(sorted(split_dir.glob('*.csv')))
    candidates.extend(sorted((unpack_dir / 'DeepMzyme_Data' / 'DeepMzyme_Colab_Bundles').glob(f'**/*{split_name}*.csv')))
    for candidate in candidates:
        if candidate.exists() and has_required_summary_columns(candidate):
            return candidate.resolve()
    found = [str(path) for path in candidates if path.exists()]
    raise FileNotFoundError(
        f'No MAHOMES-style {split_name} summary CSV found. Checked: {found}. '
        'If your bundle only contains structure_name/ec_numbers/metal_type CSVs, rebuild it with the site-level summary CSV included.'
    )

dataset_root = find_dataset_root(unpack_dir, DATASET_NAME)
train_dir = Path(TRAIN_DIR_OVERRIDE).expanduser().resolve() if TRAIN_DIR_OVERRIDE else dataset_root / 'train'
test_dir = Path(TEST_DIR_OVERRIDE).expanduser().resolve() if TEST_DIR_OVERRIDE else dataset_root / 'test'
train_csv = Path(TRAIN_CSV_OVERRIDE).expanduser().resolve() if TRAIN_CSV_OVERRIDE else find_summary_csv(train_dir, 'train')
test_csv = Path(TEST_CSV_OVERRIDE).expanduser().resolve() if TEST_CSV_OVERRIDE else find_summary_csv(test_dir, 'test')

for label, path in [('train_dir', train_dir), ('test_dir', test_dir), ('train_csv', train_csv), ('test_csv', test_csv)]:
    if not path.exists():
        raise FileNotFoundError(f'{label} does not exist: {path}')

train_structures = sorted(list(train_dir.glob('*.pdb')) + list(train_dir.glob('*.cif')) + list(train_dir.glob('*.mmcif')))
test_structures = sorted(list(test_dir.glob('*.pdb')) + list(test_dir.glob('*.cif')) + list(test_dir.glob('*.mmcif')))
if not train_structures or not test_structures:
    raise ValueError(f'Expected structure files in both train and test directories: {train_dir}, {test_dir}')

print('Dataset root:', dataset_root)
print('Train structures:', len(train_structures), train_dir)
print('Test structures:', len(test_structures), test_dir)
print('Train summary CSV:', train_csv)
print('Test summary CSV:', test_csv)

## 6. Select Single-Run Settings

Baseline-first order is: Only-GVP, Only-ESM, GVP + late fusion, then GVP + early fusion. Cross-modal attention is a more complex GVP+ESM fusion option; enable it after simpler baselines if you want residue-level structural states to attend over residue-level ESM states. The metal collapsed-4 option uses the current CLI support for collapsed-4 validation/test metrics; it does not create a separate 4-class training head.

These settings define the preserved single-run workflow. Sweep mode below reuses the same CLI-building helper and expands selected lists into multiple runs.

In [ ]:
#@title Single-run training configuration
from datetime import datetime
from pathlib import Path

TASK_MODE = 'metal_6_class'  #@param ['metal_6_class', 'metal_collapsed4_metric', 'ec_prediction']
MODEL_PRESET = 'Only-GVP'  #@param ['Only-GVP', 'Only-ESM', 'GVP + late fusion', 'GVP + early fusion', 'GVP + cross-modal attention']

EPOCHS = 50  #@param {type:"integer"}
BATCH_SIZE = 8  #@param {type:"integer"}
LEARNING_RATE = 3e-4  #@param {type:"number"}
WEIGHT_DECAY = 1e-4  #@param {type:"number"}
SEED = 42  #@param {type:"integer"}
VAL_FRACTION = 0.15  #@param {type:"number"}
DEVICE = 'cuda'  #@param ['cuda', 'cpu']
RUN_NAME = ''  #@param {type:"string"}

EC_LABEL_DEPTH = 1  #@param {type:"integer"}
EC_CONTRASTIVE_WEIGHT = 0.0  #@param {type:"number"}
NODE_FEATURE_SET = 'conservative'  #@param ['conservative']
SPLIT_BY = 'pdbid'  #@param ['pdbid', 'pdbid_chain', 'structure_id', 'pocket_id']
LR_SCHEDULE = 'fixed'  #@param ['fixed', 'cosine', 'step']
LR_STEP_SIZE = 10  #@param {type:"integer"}
LR_DECAY_GAMMA = 0.5  #@param {type:"number"}

CROSS_ATTENTION_LAYERS = 1  #@param {type:"integer"}
CROSS_ATTENTION_HEADS = 4  #@param {type:"integer"}
CROSS_ATTENTION_DROPOUT = 0.1  #@param {type:"number"}
CROSS_ATTENTION_NEIGHBORHOOD = 'all'  #@param ['all', 'first_shell', 'first_second_shell']
CROSS_ATTENTION_BIDIRECTIONAL = False  #@param {type:"boolean"}
SINGLE_CROSS_ATTENTION_LAYERS = CROSS_ATTENTION_LAYERS
SINGLE_CROSS_ATTENTION_HEADS = CROSS_ATTENTION_HEADS

USE_ESM_EMBEDDINGS = True  #@param {type:"boolean"}
ESM_EMBEDDINGS_DIR = '/content/drive/MyDrive/DeepMzyme/DeepMzyme_Data/embeddings'  #@param {type:"string"}
PREPARE_MISSING_ESM_EMBEDDINGS = False  #@param {type:"boolean"}
ALLOW_MISSING_ESM_EMBEDDINGS = False  #@param {type:"boolean"}
ALLOW_MISSING_EXTERNAL_FEATURES = True  #@param {type:"boolean"}

RING_EDGES_MODE = 'radius_only'  #@param ['radius_only', 'radius_plus_precomputed_ring', 'generate_missing_ring']
RING_EXE_PATH = 'DeepMzyme_Data/ring-4.0/out/bin/ring'  #@param {type:"string"}
RING_EDGE_OUTPUT_DIR = '/content/drive/MyDrive/DeepMzyme/DeepMzyme_Data/embeddings'  #@param {type:"string"}
PRECOMPUTED_RING_EDGES_DIR = '/content/drive/MyDrive/DeepMzyme/DeepMzyme_Data/embeddings'  #@param {type:"string"}
REQUIRE_RING_EDGES = False  #@param {type:"boolean"}
PREPARE_MISSING_RING_EDGES = False  #@param {type:"boolean"}

RUN_HELD_OUT_TEST_EVAL = True  #@param {type:"boolean"}

task_map = {
    'metal_6_class': ('metal', 'val_metal_balanced_acc'),
    'metal_collapsed4_metric': ('metal', 'val_metal_collapsed4_balanced_acc'),
    'ec_prediction': ('ec', 'val_ec_group_balanced_acc'),
}
preset_map = {
    'Only-GVP': {'model_architecture': 'only_gvp', 'uses_esm': False},
    'Only-ESM': {'model_architecture': 'only_esm', 'uses_esm': True},
    'GVP + late fusion': {'model_architecture': 'gvp', 'fusion_mode': 'late_fusion', 'uses_esm': True},
    'GVP + early fusion': {'model_architecture': 'gvp', 'fusion_mode': 'early_fusion', 'early_esm_dim': 32, 'early_esm_dropout': 0.2, 'uses_esm': True},
    'GVP + cross-modal attention': {'model_architecture': 'gvp', 'fusion_mode': 'cross_modal_attention', 'uses_esm': True},
}

task_name, selection_metric = task_map[TASK_MODE]
preset = preset_map[MODEL_PRESET]
print('Task:', task_name)
print('Selection metric:', selection_metric)
print('Model preset:', MODEL_PRESET)
print('Selected model uses ESM:', bool(preset.get('uses_esm')))
print('ESM embeddings enabled for ESM presets:', USE_ESM_EMBEDDINGS)
print('RING edge mode:', RING_EDGES_MODE)
print('Requested run name:', RUN_NAME or '(auto)')


## 7. Configure Multi-Run Sweep Mode

Sweep mode expands the selected lists into planned experiments with `itertools.product`, then runs each plan through `src/train.py`. Use validation metrics first when comparing sweep results; held-out test metrics are final reporting for the validation-selected checkpoint only and should not be used to choose hyperparameters.

In [ ]:
#@title Sweep configuration
# Keep these lists short until the full workflow is verified in Colab.
TASK_MODES = ['metal_6_class']
MODEL_PRESETS = ['Only-GVP']
RING_EDGES_MODES = ['radius_only']
LEARNING_RATES = [3e-4]
WEIGHT_DECAYS = [1e-4]
BATCH_SIZES = [8]
SEEDS = [42]
NODE_FEATURE_SETS = ['conservative']
EC_LABEL_DEPTHS = [1]
EC_CONTRASTIVE_WEIGHTS = [0.0]
CROSS_ATTENTION_LAYERS = [1]
CROSS_ATTENTION_HEADS = [4]
LR_SCHEDULES = ['fixed']

MAX_SWEEP_RUNS = 24  #@param {type:"integer"}
STOP_ON_FIRST_FAILURE = True  #@param {type:"boolean"}
SKIP_EXISTING_RUNS = True  #@param {type:"boolean"}

print('Sweep task modes:', TASK_MODES)
print('Sweep model presets:', MODEL_PRESETS)
print('Sweep RING edge modes:', RING_EDGES_MODES)
print('Maximum planned runs:', MAX_SWEEP_RUNS)


## 8. Build Commands and Planned Experiments

Review the single-run command and the sweep plan before running. The helper functions below build CLI commands only; model training still happens exclusively in `src/train.py`.

In [ ]:
import csv
import hashlib
import itertools
import json
import os
import re
import shlex
import subprocess
import sys
from datetime import datetime
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

local_runs_dir.mkdir(parents=True, exist_ok=True)

VALID_RING_EDGE_MODES = {'radius_only', 'radius_plus_precomputed_ring', 'generate_missing_ring'}


def slugify(value):
    text = str(value).lower().replace('+', 'plus')
    text = re.sub(r'[^a-z0-9]+', '_', text).strip('_')
    return text or 'value'


def format_float_token(value):
    token = f'{float(value):.0e}' if abs(float(value)) < 0.001 else f'{float(value):g}'
    return token.replace('+', '').replace('-', 'm').replace('.', 'p')


def config_hash(config):
    payload = json.dumps(config, sort_keys=True, default=str)
    return hashlib.sha1(payload.encode('utf-8')).hexdigest()[:8]


def command_string(cmd):
    return ' '.join(shlex.quote(str(part)) for part in cmd)


def make_safe_run_name(config, requested_name=None):
    if requested_name:
        return slugify(requested_name)
    parts = [
        slugify(config['task_mode']),
        slugify(config['model_preset']),
        f"edges{slugify(config['ring_edges_mode'])}",
        f"lr{format_float_token(config['learning_rate'])}",
        f"wd{format_float_token(config['weight_decay'])}",
        f"bs{config['batch_size']}",
        f"seed{config['seed']}",
        f"node{slugify(config['node_feature_set'])}",
    ]
    if config.get('fusion_mode'):
        parts.insert(2, slugify(config['fusion_mode']))
    if config['task'] == 'ec':
        parts.extend([
            f"ecdepth{config['ec_label_depth']}",
            f"eccont{format_float_token(config['ec_contrastive_weight'])}",
        ])
    if config.get('fusion_mode') == 'cross_modal_attention':
        parts.extend([
            f"xal{config['cross_attention_layers']}",
            f"xah{config['cross_attention_heads']}",
        ])
    parts.append(f"h{config_hash(config)}")
    return '_'.join(parts)


def path_from_control(value):
    return Path(str(value)).expanduser()


def resolve_ring_exe_control(value):
    ring_path = path_from_control(value)
    if ring_path.is_absolute():
        return ring_path
    parts = ring_path.parts
    if parts and parts[0] == drive_data_dir.name:
        drive_candidate = drive_data_dir.parent.joinpath(*parts)
        repo_candidate = repo_dir / ring_path
    else:
        drive_candidate = drive_data_dir / ring_path
        repo_candidate = repo_dir / drive_data_dir.name / ring_path
    drive_mount_available = str(drive_data_dir).startswith('/content/drive/') and Path('/content/drive').exists()
    if drive_mount_available or drive_data_dir.exists():
        return drive_candidate
    if repo_candidate.exists() or (repo_dir / drive_data_dir.name).exists():
        return repo_candidate
    return drive_candidate if str(drive_data_dir).startswith('/content/drive/') else repo_candidate


def sample_files(root, pattern, limit=5):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(root.rglob(pattern))[:limit]


def count_files(root, pattern):
    root = Path(root)
    if not root.exists():
        return 0
    return sum(1 for _ in root.rglob(pattern))


def build_run_config(
    *,
    task_mode,
    model_preset,
    ring_edges_mode,
    learning_rate,
    weight_decay,
    batch_size,
    seed,
    node_feature_set,
    ec_label_depth,
    ec_contrastive_weight,
    cross_attention_layers,
    cross_attention_heads,
    lr_schedule,
    requested_name=None,
):
    if ring_edges_mode not in VALID_RING_EDGE_MODES:
        raise ValueError(f'Unsupported RING_EDGES_MODE: {ring_edges_mode}')
    task, selection_metric = task_map[task_mode]
    preset = dict(preset_map[model_preset])
    uses_esm = bool(preset.get('uses_esm'))
    esm_embeddings_dir = str(path_from_control(ESM_EMBEDDINGS_DIR)) if uses_esm else ''
    ring_output_dir = str(path_from_control(RING_EDGE_OUTPUT_DIR))
    precomputed_ring_dir = str(path_from_control(PRECOMPUTED_RING_EDGES_DIR))
    ring_features_dir = precomputed_ring_dir if ring_edges_mode == 'radius_plus_precomputed_ring' else ring_output_dir
    prepare_ring = bool(PREPARE_MISSING_RING_EDGES and ring_edges_mode == 'generate_missing_ring')
    require_ring = bool(REQUIRE_RING_EDGES and ring_edges_mode != 'radius_only')
    config = {
        'task_mode': task_mode,
        'task': task,
        'selection_metric': selection_metric,
        'model_preset': model_preset,
        'model_architecture': preset['model_architecture'],
        'fusion_mode': preset.get('fusion_mode'),
        'early_esm_dim': preset.get('early_esm_dim'),
        'early_esm_dropout': preset.get('early_esm_dropout'),
        'epochs': int(EPOCHS),
        'batch_size': int(batch_size),
        'learning_rate': float(learning_rate),
        'weight_decay': float(weight_decay),
        'seed': int(seed),
        'val_fraction': float(VAL_FRACTION),
        'device': DEVICE,
        'split_by': SPLIT_BY,
        'node_feature_set': node_feature_set,
        'ec_label_depth': int(ec_label_depth),
        'ec_contrastive_weight': float(ec_contrastive_weight),
        'lr_schedule': lr_schedule,
        'lr_step_size': int(LR_STEP_SIZE),
        'lr_decay_gamma': float(LR_DECAY_GAMMA),
        'cross_attention_layers': int(cross_attention_layers),
        'cross_attention_heads': int(cross_attention_heads),
        'cross_attention_dropout': float(CROSS_ATTENTION_DROPOUT),
        'cross_attention_neighborhood': CROSS_ATTENTION_NEIGHBORHOOD,
        'cross_attention_bidirectional': bool(CROSS_ATTENTION_BIDIRECTIONAL),
        'run_test_eval': bool(RUN_HELD_OUT_TEST_EVAL),
        'uses_esm': uses_esm,
        'use_esm_embeddings': bool(USE_ESM_EMBEDDINGS and uses_esm),
        'esm_embeddings_dir': esm_embeddings_dir,
        'prepare_missing_esm_embeddings': bool(PREPARE_MISSING_ESM_EMBEDDINGS and uses_esm),
        'allow_missing_esm_embeddings': bool(ALLOW_MISSING_ESM_EMBEDDINGS or not uses_esm),
        'allow_missing_external_features': bool(ALLOW_MISSING_EXTERNAL_FEATURES),
        'ring_edges_mode': ring_edges_mode,
        'require_ring_edges': require_ring,
        'prepare_missing_ring_edges': prepare_ring,
        'ring_exe_path': str(resolve_ring_exe_control(RING_EXE_PATH)) if ring_edges_mode == 'generate_missing_ring' else '',
        'ring_edge_output_dir': ring_output_dir,
        'precomputed_ring_edges_dir': precomputed_ring_dir if ring_edges_mode == 'radius_plus_precomputed_ring' else '',
        'ring_features_dir': ring_features_dir,
    }
    config['run_name'] = make_safe_run_name(config, requested_name=requested_name)
    config['run_dir'] = local_runs_dir / config['run_name']
    return config


def esm_readiness_issues(config):
    if not config['uses_esm']:
        return []
    issues = []
    embeddings_dir = Path(config['esm_embeddings_dir'])
    embedding_count = count_files(embeddings_dir, '*_esmc.pt')
    if not config['use_esm_embeddings'] and not config['allow_missing_esm_embeddings']:
        issues.append('Selected ESM preset has USE_ESM_EMBEDDINGS=False. Set ALLOW_MISSING_ESM_EMBEDDINGS=True to train explicitly without embeddings.')
    if not config['prepare_missing_esm_embeddings'] and not config['allow_missing_esm_embeddings']:
        if not embeddings_dir.exists():
            issues.append(f'ESM embeddings directory does not exist: {embeddings_dir}')
        elif embedding_count == 0:
            issues.append(f'No *_esmc.pt files found under ESM embeddings directory: {embeddings_dir}')
    return issues


def ring_readiness_issues(config):
    issues = []
    mode = config['ring_edges_mode']
    if mode == 'radius_only':
        return issues
    ring_dir = Path(config['ring_features_dir'])
    ring_count = count_files(ring_dir, '*ringEdges')
    if mode == 'radius_plus_precomputed_ring' and config['require_ring_edges']:
        if not ring_dir.exists():
            issues.append(f'Precomputed RING edge directory does not exist: {ring_dir}')
        elif ring_count == 0:
            issues.append(f'No *ringEdges files found under precomputed RING edge directory: {ring_dir}')
    if mode == 'generate_missing_ring' and config['prepare_missing_ring_edges']:
        ring_exe = Path(config['ring_exe_path'])
        if not ring_exe.is_file():
            issues.append(f'RING executable does not exist: {ring_exe}')
        elif not os.access(ring_exe, os.X_OK):
            issues.append(f'RING executable is not executable: {ring_exe}. Run: chmod +x {ring_exe}')
    return issues


def readiness_issues(config):
    return esm_readiness_issues(config) + ring_readiness_issues(config)


def validate_run_before_training(config):
    issues = readiness_issues(config)
    if issues:
        raise RuntimeError('Pre-training configuration check failed:\n- ' + '\n- '.join(issues))


def print_run_checks(config, *, label):
    embeddings_dir = Path(config['esm_embeddings_dir']) if config['esm_embeddings_dir'] else None
    embedding_files = sample_files(embeddings_dir, '*_esmc.pt') if embeddings_dir is not None else []
    ring_features_dir = Path(config['ring_features_dir'])
    ring_files = sample_files(ring_features_dir, '*ringEdges')
    ring_exe_user_path = path_from_control(RING_EXE_PATH)
    ring_exe = Path(config['ring_exe_path']) if config['ring_exe_path'] else resolve_ring_exe_control(RING_EXE_PATH)
    issues = readiness_issues(config)
    print(f'{label} checks')
    print('  selected model preset:', config['model_preset'])
    print('  selected model uses ESM:', config['uses_esm'])
    print('  ESM embeddings directory:', config['esm_embeddings_dir'] or 'not used by this preset')
    print('  ESM embeddings directory exists:', embeddings_dir.exists() if embeddings_dir is not None else False)
    print('  existing ESM embedding files:', count_files(embeddings_dir, '*_esmc.pt') if embeddings_dir is not None else 0)
    print('  sample ESM embedding files:', [str(path) for path in embedding_files[:3]])
    print('  missing ESM embeddings will be generated:', config['prepare_missing_esm_embeddings'])
    print('  missing ESM embeddings are allowed:', config['allow_missing_esm_embeddings'])
    print('  --esm-embeddings-dir is being passed:', config['uses_esm'])
    print('  --allow-missing-esm-embeddings is being passed:', config['allow_missing_esm_embeddings'])
    print('  --no-prepare-missing-esm-embeddings is being passed:', (not config['prepare_missing_esm_embeddings']) or not config['uses_esm'])
    print('  RING_EDGES_MODE:', config['ring_edges_mode'])
    print('  RING_EXE_PATH user setting:', str(ring_exe_user_path))
    print('  RING_EXE_PATH resolved:', str(ring_exe))
    print('  RING_EXE_PATH exists:', ring_exe.is_file())
    print('  RING_EXE_PATH executable:', os.access(ring_exe, os.X_OK) if ring_exe.exists() else False)
    print('  RING edge files expected under:', ring_features_dir)
    print('  generated RING edge files will be written under:', config['ring_edge_output_dir'])
    print('  existing RING edge files:', count_files(ring_features_dir, '*ringEdges'))
    print('  sample RING edge files:', [str(path) for path in ring_files[:3]])
    print('  precomputed RING edge files are being used:', config['ring_edges_mode'] == 'radius_plus_precomputed_ring')
    print('  missing RING edges will be generated:', config['prepare_missing_ring_edges'])
    print('  --require-ring-edges is being passed:', config['require_ring_edges'])
    print('  --prepare-missing-ring-edges is being passed:', config['prepare_missing_ring_edges'])
    if issues:
        print('  pre-training warnings/errors:')
        for issue in issues:
            print('   -', issue)


def build_train_command(config):
    cmd = [
        sys.executable, str(repo_dir / 'src' / 'train.py'),
        '--task', config['task'],
        '--structure-dir', str(train_dir),
        '--summary-csv', str(train_csv),
        '--runs-dir', str(local_runs_dir),
        '--run-name', config['run_name'],
        '--model-architecture', config['model_architecture'],
        '--epochs', str(config['epochs']),
        '--batch-size', str(config['batch_size']),
        '--learning-rate', str(config['learning_rate']),
        '--weight-decay', str(config['weight_decay']),
        '--seed', str(config['seed']),
        '--val-fraction', str(config['val_fraction']),
        '--device', config['device'],
        '--split-by', config['split_by'],
        '--selection-metric', config['selection_metric'],
        '--node-feature-set', config['node_feature_set'],
        '--lr-schedule', config['lr_schedule'],
    ]
    if config['lr_schedule'] == 'step':
        cmd.extend(['--lr-step-size', str(config['lr_step_size']), '--lr-decay-gamma', str(config['lr_decay_gamma'])])
    if config.get('fusion_mode'):
        cmd.extend(['--fusion-mode', config['fusion_mode']])
    if config.get('fusion_mode') == 'early_fusion':
        cmd.extend(['--use-early-esm', '--early-esm-dim', str(config['early_esm_dim']), '--early-esm-dropout', str(config['early_esm_dropout'])])
    if config.get('fusion_mode') == 'cross_modal_attention':
        cmd.extend([
            '--cross-attention-layers', str(config['cross_attention_layers']),
            '--cross-attention-heads', str(config['cross_attention_heads']),
            '--cross-attention-dropout', str(config['cross_attention_dropout']),
            '--cross-attention-neighborhood', config['cross_attention_neighborhood'],
        ])
        if config['cross_attention_bidirectional']:
            cmd.append('--cross-attention-bidirectional')
    if config['task'] == 'ec':
        cmd.extend(['--ec-label-depth', str(config['ec_label_depth'])])
        cmd.extend(['--ec-contrastive-weight', str(config['ec_contrastive_weight'])])
    if config['run_test_eval']:
        cmd.extend(['--test-structure-dir', str(test_dir), '--test-summary-csv', str(test_csv), '--run-test-eval'])
    if config['uses_esm']:
        cmd.extend(['--esm-embeddings-dir', config['esm_embeddings_dir']])
        if config['allow_missing_esm_embeddings']:
            cmd.append('--allow-missing-esm-embeddings')
        if not config['prepare_missing_esm_embeddings']:
            cmd.append('--no-prepare-missing-esm-embeddings')
    else:
        cmd.extend(['--allow-missing-esm-embeddings', '--no-prepare-missing-esm-embeddings'])
    if config['allow_missing_external_features']:
        cmd.append('--allow-missing-external-features')
    if config['require_ring_edges']:
        cmd.append('--require-ring-edges')
    if config['prepare_missing_ring_edges']:
        cmd.append('--prepare-missing-ring-edges')
    return cmd


def stream_command(cmd, *, cwd, env):
    process = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    return process.wait()


SWEEP_STATUS_COLUMNS = [
    'run_index', 'total_runs', 'run_name', 'task', 'task_mode', 'model_preset',
    'model_architecture', 'fusion_mode', 'learning_rate', 'weight_decay',
    'batch_size', 'seed', 'node_feature_set', 'ec_label_depth',
    'ec_contrastive_weight', 'cross_attention_layers', 'cross_attention_heads',
    'lr_schedule', 'selection_metric', 'val_fraction', 'split_by', 'uses_esm',
    'esm_embeddings_dir', 'prepare_missing_esm_embeddings', 'allow_missing_esm_embeddings',
    'ring_edges_mode', 'require_ring_edges', 'prepare_missing_ring_edges',
    'ring_exe_path', 'ring_edge_output_dir', 'ring_features_dir',
    'status', 'start_time', 'end_time', 'update_time', 'run_dir',
    'selected_checkpoint', 'selected_metric_value', 'test_summary', 'command', 'error_message',
]


def write_sweep_status(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=SWEEP_STATUS_COLUMNS)
        writer.writeheader()
        for row in rows:
            writer.writerow({column: row.get(column, '') for column in SWEEP_STATUS_COLUMNS})


def summarize_completed_run(run_dir):
    def read_json(path):
        try:
            return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}
        except Exception as exc:
            return {'_read_error': str(exc)}
    run_config = read_json(run_dir / 'run_config.json')
    run_metadata = read_json(run_dir / 'run_metadata.json')
    test_report = read_json(run_dir / 'test_report.json')
    selected_checkpoint = run_metadata.get('selected_checkpoint') or run_config.get('selected_checkpoint')
    selection_metric = run_metadata.get('selection_metric') or run_config.get('selection_metric')
    selected_metric_value = run_metadata.get('selected_metric_value') or run_config.get('selected_metric_value')
    metrics = test_report.get('metrics', {}) if isinstance(test_report.get('metrics'), dict) else {}
    priority = [
        'test_metal_balanced_acc', 'test_metal_collapsed4_balanced_acc',
        'test_ec_group_balanced_acc', 'test_ec_group_level_1_balanced_acc',
        'test_ec_level_1_balanced_acc', 'test_loss',
    ]
    test_summary = {key: metrics[key] for key in priority if key in metrics}
    return {
        'selected_checkpoint': selected_checkpoint,
        'selection_metric': selection_metric,
        'selected_metric_value': selected_metric_value,
        'test_summary': test_summary,
    }


def run_report(runs_dir, out_csv, out_figure):
    report_cmd = [
        sys.executable, str(repo_dir / 'src' / 'report_runs.py'),
        '--runs-dir', str(runs_dir),
        '--out-csv', str(out_csv),
        '--out-figure', str(out_figure),
    ]
    print('Report command:')
    print(command_string(report_cmd))
    return stream_command(report_cmd, cwd=repo_dir, env=base_env)


def build_sweep_runs():
    planned = []
    base_product = itertools.product(
        TASK_MODES,
        MODEL_PRESETS,
        RING_EDGES_MODES,
        LEARNING_RATES,
        WEIGHT_DECAYS,
        BATCH_SIZES,
        SEEDS,
        NODE_FEATURE_SETS,
        LR_SCHEDULES,
    )
    seen_names = set()
    for task_mode, model_preset, ring_edges_mode, lr, wd, batch_size, seed, node_features, lr_schedule in base_product:
        ec_depth_values = EC_LABEL_DEPTHS if task_mode == 'ec_prediction' else [EC_LABEL_DEPTH]
        ec_weight_values = EC_CONTRASTIVE_WEIGHTS if task_mode == 'ec_prediction' else [EC_CONTRASTIVE_WEIGHT]
        cross_layer_values = CROSS_ATTENTION_LAYERS if model_preset == 'GVP + cross-modal attention' else [CROSS_ATTENTION_LAYERS[0]]
        cross_head_values = CROSS_ATTENTION_HEADS if model_preset == 'GVP + cross-modal attention' else [CROSS_ATTENTION_HEADS[0]]
        for ec_depth, ec_weight, cross_layers, cross_heads in itertools.product(
            ec_depth_values,
            ec_weight_values,
            cross_layer_values,
            cross_head_values,
        ):
            config = build_run_config(
                task_mode=task_mode,
                model_preset=model_preset,
                ring_edges_mode=ring_edges_mode,
                learning_rate=lr,
                weight_decay=wd,
                batch_size=batch_size,
                seed=seed,
                node_feature_set=node_features,
                ec_label_depth=ec_depth,
                ec_contrastive_weight=ec_weight,
                cross_attention_layers=cross_layers,
                cross_attention_heads=cross_heads,
                lr_schedule=lr_schedule,
            )
            if config['run_name'] in seen_names:
                continue
            seen_names.add(config['run_name'])
            config['command'] = build_train_command(config)
            planned.append(config)
    return planned


def planned_table_rows(planned_runs):
    rows = []
    for index, run in enumerate(planned_runs, start=1):
        rows.append({
            'run_index': index,
            'run_name': run['run_name'],
            'task': run['task'],
            'task_mode': run['task_mode'],
            'model_preset': run['model_preset'],
            'model_architecture': run['model_architecture'],
            'fusion_mode': run.get('fusion_mode') or 'none',
            'ring_edges_mode': run['ring_edges_mode'],
            'uses_esm': run['uses_esm'],
            'esm_embeddings_dir': run['esm_embeddings_dir'] or 'NA',
            'prepare_missing_esm_embeddings': run['prepare_missing_esm_embeddings'],
            'allow_missing_esm_embeddings': run['allow_missing_esm_embeddings'],
            'require_ring_edges': run['require_ring_edges'],
            'prepare_missing_ring_edges': run['prepare_missing_ring_edges'],
            'ring_edge_output_dir': run['ring_edge_output_dir'],
            'learning_rate': run['learning_rate'],
            'weight_decay': run['weight_decay'],
            'batch_size': run['batch_size'],
            'seed': run['seed'],
            'node_feature_set': run['node_feature_set'],
            'ec_label_depth': run['ec_label_depth'] if run['task'] == 'ec' else 'NA',
            'ec_contrastive_weight': run['ec_contrastive_weight'] if run['task'] == 'ec' else 'NA',
            'cross_attention_layers': run['cross_attention_layers'] if run.get('fusion_mode') == 'cross_modal_attention' else 'NA',
            'cross_attention_heads': run['cross_attention_heads'] if run.get('fusion_mode') == 'cross_modal_attention' else 'NA',
            'selection_metric': run['selection_metric'],
            'run_dir': str(run['run_dir']),
        })
    return rows


def status_row_for_run(index, total_runs, run):
    return {
        'run_index': index,
        'total_runs': total_runs,
        'run_name': run['run_name'],
        'task': run['task'],
        'task_mode': run['task_mode'],
        'model_preset': run['model_preset'],
        'model_architecture': run['model_architecture'],
        'fusion_mode': run.get('fusion_mode') or 'none',
        'learning_rate': run['learning_rate'],
        'weight_decay': run['weight_decay'],
        'batch_size': run['batch_size'],
        'seed': run['seed'],
        'node_feature_set': run['node_feature_set'],
        'ec_label_depth': run['ec_label_depth'] if run['task'] == 'ec' else 'NA',
        'ec_contrastive_weight': run['ec_contrastive_weight'] if run['task'] == 'ec' else 'NA',
        'cross_attention_layers': run['cross_attention_layers'] if run.get('fusion_mode') == 'cross_modal_attention' else 'NA',
        'cross_attention_heads': run['cross_attention_heads'] if run.get('fusion_mode') == 'cross_modal_attention' else 'NA',
        'lr_schedule': run['lr_schedule'],
        'selection_metric': run['selection_metric'],
        'val_fraction': run['val_fraction'],
        'split_by': run['split_by'],
        'uses_esm': run['uses_esm'],
        'esm_embeddings_dir': run['esm_embeddings_dir'] or 'NA',
        'prepare_missing_esm_embeddings': run['prepare_missing_esm_embeddings'],
        'allow_missing_esm_embeddings': run['allow_missing_esm_embeddings'],
        'ring_edges_mode': run['ring_edges_mode'],
        'require_ring_edges': run['require_ring_edges'],
        'prepare_missing_ring_edges': run['prepare_missing_ring_edges'],
        'ring_exe_path': run['ring_exe_path'] if run['prepare_missing_ring_edges'] else 'NA',
        'ring_edge_output_dir': run['ring_edge_output_dir'],
        'ring_features_dir': run['ring_features_dir'],
        'status': 'planned',
        'start_time': '',
        'end_time': '',
        'update_time': datetime.now().isoformat(timespec='seconds'),
        'run_dir': str(run['run_dir']),
        'selected_checkpoint': '',
        'selected_metric_value': '',
        'test_summary': '',
        'command': command_string(run['command']),
        'error_message': '',
    }


def mark_status(row, status):
    row['status'] = status
    row['update_time'] = datetime.now().isoformat(timespec='seconds')


single_run_config = build_run_config(
    task_mode=TASK_MODE,
    model_preset=MODEL_PRESET,
    ring_edges_mode=RING_EDGES_MODE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    seed=SEED,
    node_feature_set=NODE_FEATURE_SET,
    ec_label_depth=EC_LABEL_DEPTH,
    ec_contrastive_weight=EC_CONTRASTIVE_WEIGHT,
    cross_attention_layers=SINGLE_CROSS_ATTENTION_LAYERS,
    cross_attention_heads=SINGLE_CROSS_ATTENTION_HEADS,
    lr_schedule=LR_SCHEDULE,
    requested_name=RUN_NAME or None,
)
if single_run_config['run_dir'].exists() and not RUN_NAME:
    collision_suffix = datetime.now().strftime('%Y%m%d_%H%M%S')
    single_run_config['run_name'] = f"{single_run_config['run_name']}_{collision_suffix}"
    single_run_config['run_dir'] = local_runs_dir / single_run_config['run_name']
train_cmd = build_train_command(single_run_config)
single_run_config['command'] = train_cmd
target_run_dir = single_run_config['run_dir']
RUN_NAME = single_run_config['run_name']

planned_sweep_runs = build_sweep_runs()
if len(planned_sweep_runs) > MAX_SWEEP_RUNS:
    raise ValueError(f'Sweep would run {len(planned_sweep_runs)} experiments, above MAX_SWEEP_RUNS={MAX_SWEEP_RUNS}.')

planned_sweep_table = planned_table_rows(planned_sweep_runs)
sweep_status_path = local_runs_dir / 'sweep_status.csv'
final_comparison_csv = local_runs_dir / 'sweep_comparison.csv'
final_comparison_figure = local_runs_dir / 'sweep_comparison.png'
completed_run_dirs = []
failed_run_dirs = []

os.environ['RING_EXE_PATH'] = str(resolve_ring_exe_control(RING_EXE_PATH))
os.environ['EMBEDDINGS_DIR'] = single_run_config['ring_features_dir']
base_env = os.environ.copy()
base_env['PYTHONPATH'] = str(repo_dir / 'src') + os.pathsep + base_env.get('PYTHONPATH', '')


def env_for_run(config):
    run_env = base_env.copy()
    run_env['RING_EXE_PATH'] = config['ring_exe_path'] or str(resolve_ring_exe_control(RING_EXE_PATH))
    run_env['EMBEDDINGS_DIR'] = config['ring_features_dir']
    return run_env


print_run_checks(single_run_config, label='Single-run')
print('\nSingle-run command:')
print(command_string(train_cmd))
print('Single-run output directory:', target_run_dir)
print('\nTotal planned sweep runs:', len(planned_sweep_runs))
print('Sweep status path:', sweep_status_path)
print('Sweep comparison CSV:', final_comparison_csv)
print('Sweep output directory:', local_runs_dir)
if pd is not None:
    display(pd.DataFrame(planned_sweep_table))
else:
    for row in planned_sweep_table:
        print(row)


## 9. Run Single Training or Sweep

This is the only section that starts training. Keep both run toggles disabled until paths, commands, and planned experiments have been checked. If both modes are enabled accidentally, the notebook stops unless `ALLOW_SINGLE_AND_SWEEP=True`.

In [ ]:
#@title Execute training or sweep
RUN_SINGLE_MODE = False  #@param {type:"boolean"}
RUN_SWEEP_MODE = False  #@param {type:"boolean"}
ALLOW_SINGLE_AND_SWEEP = False  #@param {type:"boolean"}

if RUN_SINGLE_MODE and RUN_SWEEP_MODE and not ALLOW_SINGLE_AND_SWEEP:
    raise ValueError('RUN_SINGLE_MODE and RUN_SWEEP_MODE are both enabled. Disable one or set ALLOW_SINGLE_AND_SWEEP=True.')
if DEVICE == 'cuda':
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('DEVICE is cuda, but torch.cuda.is_available() is False. Change DEVICE to cpu or enable a GPU runtime.')
if not RUN_SINGLE_MODE and not RUN_SWEEP_MODE:
    print('Training skipped. Enable RUN_SINGLE_MODE or RUN_SWEEP_MODE when ready.')

if RUN_SINGLE_MODE:
    validate_run_before_training(single_run_config)
    print('=' * 60)
    print('Starting single run')
    print('Run name:', single_run_config['run_name'])
    print('Task:', single_run_config['task'])
    print('Model:', single_run_config['model_preset'])
    print('RING edge mode:', single_run_config['ring_edges_mode'])
    print('Uses ESM:', single_run_config['uses_esm'])
    print('Learning rate:', single_run_config['learning_rate'])
    print('Selection metric:', single_run_config['selection_metric'])
    print('Output directory:', single_run_config['run_dir'])
    print('=' * 60)
    return_code = stream_command(single_run_config['command'], cwd=repo_dir, env=env_for_run(single_run_config))
    if return_code != 0:
        failed_run_dirs.append(single_run_config['run_dir'])
        print(f'Single run failed with exit code {return_code}: {single_run_config["run_dir"]}')
    else:
        completed_run_dirs.append(single_run_config['run_dir'])
        summary = summarize_completed_run(single_run_config['run_dir'])
        print('Single run succeeded:', single_run_config['run_dir'])
        print('Selected checkpoint:', summary.get('selected_checkpoint'))
        print('Best validation metric:', summary.get('selection_metric'), summary.get('selected_metric_value'))
        print('Held-out test metric summary:', summary.get('test_summary'))

if RUN_SWEEP_MODE:
    sweep_status_rows = []
    total_runs = len(planned_sweep_runs)
    for index, run in enumerate(planned_sweep_runs, start=1):
        sweep_status_rows.append(status_row_for_run(index, total_runs, run))
    write_sweep_status(sweep_status_rows, sweep_status_path)

    print('Total planned runs:', total_runs)
    print('Sweep status CSV:', sweep_status_path)
    print('Sweep outputs directory:', local_runs_dir)
    if pd is not None:
        display(pd.DataFrame(planned_table_rows(planned_sweep_runs)))
    else:
        for row in planned_table_rows(planned_sweep_runs):
            print(row)

    for index, run in enumerate(planned_sweep_runs, start=1):
        validate_run_before_training(run)
        status_row = sweep_status_rows[index - 1]
        if SKIP_EXISTING_RUNS and run['run_dir'].exists():
            summary = summarize_completed_run(run['run_dir'])
            mark_status(status_row, 'skipped')
            status_row['end_time'] = datetime.now().isoformat(timespec='seconds')
            status_row['selected_checkpoint'] = summary.get('selected_checkpoint') or ''
            status_row['selected_metric_value'] = '' if summary.get('selected_metric_value') is None else summary.get('selected_metric_value')
            status_row['test_summary'] = json.dumps(summary.get('test_summary') or {}, sort_keys=True)
            status_row['error_message'] = 'run directory already exists'
            write_sweep_status(sweep_status_rows, sweep_status_path)
            completed_run_dirs.append(run['run_dir'])
            print(f'Skipping existing run directory: {run["run_dir"]}')
            continue

        print('=' * 60)
        print(f'Starting run {index} / {total_runs}')
        print('Run name:', run['run_name'])
        print('Task:', run['task'])
        print('Model:', run['model_preset'])
        print('RING edge mode:', run['ring_edges_mode'])
        print('Uses ESM:', run['uses_esm'])
        print('Learning rate:', run['learning_rate'])
        print('Selection metric:', run['selection_metric'])
        print('Output directory:', run['run_dir'])
        print('=' * 60)

        mark_status(status_row, 'running')
        status_row['start_time'] = datetime.now().isoformat(timespec='seconds')
        write_sweep_status(sweep_status_rows, sweep_status_path)
        return_code = stream_command(run['command'], cwd=repo_dir, env=env_for_run(run))
        status_row['end_time'] = datetime.now().isoformat(timespec='seconds')
        if return_code == 0:
            mark_status(status_row, 'succeeded')
            completed_run_dirs.append(run['run_dir'])
            summary = summarize_completed_run(run['run_dir'])
            status_row['selected_checkpoint'] = summary.get('selected_checkpoint') or ''
            status_row['selected_metric_value'] = '' if summary.get('selected_metric_value') is None else summary.get('selected_metric_value')
            status_row['test_summary'] = json.dumps(summary.get('test_summary') or {}, sort_keys=True)
            print(f'Run {index} succeeded:', run['run_dir'])
            print('Selected checkpoint:', summary.get('selected_checkpoint'))
            print('Best validation metric:', summary.get('selection_metric'), summary.get('selected_metric_value'))
            print('Held-out test metric summary:', summary.get('test_summary'))
        else:
            mark_status(status_row, 'failed')
            status_row['error_message'] = f'exit code {return_code}'
            status_row['test_summary'] = ''
            failed_run_dirs.append(run['run_dir'])
            print(f'Run {index} failed with exit code {return_code}: {run["run_dir"]}')
            if STOP_ON_FIRST_FAILURE:
                write_sweep_status(sweep_status_rows, sweep_status_path)
                raise RuntimeError(f'Stopping sweep after failed run {index}: {run["run_name"]}')
        write_sweep_status(sweep_status_rows, sweep_status_path)

    report_code = run_report(local_runs_dir, final_comparison_csv, final_comparison_figure)
    if report_code != 0:
        print(f'Warning: report_runs.py failed with exit code {report_code}')
    succeeded = sum(1 for row in sweep_status_rows if row['status'] == 'succeeded')
    skipped = sum(1 for row in sweep_status_rows if row['status'] == 'skipped')
    failed = sum(1 for row in sweep_status_rows if row['status'] == 'failed')
    print('=' * 60)
    print('Sweep finished')
    print('Succeeded:', succeeded)
    print('Skipped:', skipped)
    print('Failed:', failed)
    print('RING edge modes:', sorted({run['ring_edges_mode'] for run in planned_sweep_runs}))
    print('ESM-enabled planned runs:', sum(1 for run in planned_sweep_runs if run['uses_esm']))
    print('Sweep status CSV:', sweep_status_path)
    print('Final comparison CSV:', final_comparison_csv)
    print('Copied Drive outputs after running the copy cell:', drive_output_dir)
    print('=' * 60)


## 10. Copy Run Outputs Back to Drive

Copy local run directories and sweep artifacts to Drive after training. If `LOCAL_RUNS_DIR` already points inside Drive, this still creates a stable copy under `DRIVE_DATA_DIR/Train_Parameters_Models_and_Results`.

In [ ]:
#@title Copy outputs to Drive
COPY_OUTPUTS_TO_DRIVE = True  #@param {type:"boolean"}

import shutil

if 'completed_run_dirs' not in globals():
    completed_run_dirs = []
if 'failed_run_dirs' not in globals():
    failed_run_dirs = []

drive_runs_dir = drive_output_dir / 'runs'
drive_sweep_dir = drive_output_dir / 'sweeps'
run_dirs_to_copy = []
for candidate in completed_run_dirs + failed_run_dirs:
    candidate = Path(candidate)
    if candidate.exists() and candidate not in run_dirs_to_copy:
        run_dirs_to_copy.append(candidate)
if not run_dirs_to_copy and target_run_dir.exists():
    run_dirs_to_copy.append(target_run_dir)

if COPY_OUTPUTS_TO_DRIVE:
    drive_runs_dir.mkdir(parents=True, exist_ok=True)
    copied = []
    for run_dir in run_dirs_to_copy:
        drive_run_dir = drive_runs_dir / run_dir.name
        if drive_run_dir.exists():
            shutil.rmtree(drive_run_dir)
        shutil.copytree(run_dir, drive_run_dir)
        copied.append(drive_run_dir)
        print('Copied run output to:', drive_run_dir)

    drive_sweep_dir.mkdir(parents=True, exist_ok=True)
    for artifact in [sweep_status_path, final_comparison_csv, final_comparison_figure]:
        artifact = Path(artifact)
        if artifact.exists():
            destination = drive_sweep_dir / artifact.name
            shutil.copy2(artifact, destination)
            print('Copied sweep artifact to:', destination)
    if not copied:
        print('No completed run directories found to copy yet.')
    print('Drive output directory:', drive_output_dir)
else:
    print('Copy skipped.')

## 11. Summarize Runs with `src/report_runs.py`

The summary CSV is the main comparison table. The optional figure is created only when matplotlib is available and the selected validation metric is numeric. Sweep execution already runs this once at the end; this section lets you regenerate the report from local or copied Drive outputs.

In [ ]:
#@title Generate run summary
SUMMARY_SOURCE = 'drive_outputs'  #@param ['local_runs', 'drive_outputs']
SUMMARY_BASENAME = 'baseline_first_summary'  #@param {type:"string"}

summary_runs_dir = local_runs_dir if SUMMARY_SOURCE == 'local_runs' else drive_runs_dir
summary_csv = drive_output_dir / f'{SUMMARY_BASENAME}.csv'
summary_figure = drive_output_dir / f'{SUMMARY_BASENAME}.png'
drive_output_dir.mkdir(parents=True, exist_ok=True)

if not summary_runs_dir.exists():
    print(f'No run directory found yet: {summary_runs_dir}')
else:
    report_code = run_report(summary_runs_dir, summary_csv, summary_figure)
    if report_code != 0:
        raise RuntimeError(f'report_runs.py failed with exit code {report_code}')
    print('Summary CSV:', summary_csv)
    print('Summary figure:', summary_figure)

## 12. Display the Summary Table and Figure

Use this table to compare runs by validation-selected metrics first. Test metrics are for final held-out reporting after selecting checkpoints by validation performance.

In [ ]:
from IPython.display import Image, display
import pandas as pd

if summary_csv.exists():
    summary_df = pd.read_csv(summary_csv)
    display(summary_df)
else:
    print(f'Summary CSV not found yet: {summary_csv}')

if summary_figure.exists():
    display(Image(filename=str(summary_figure)))
else:
    print(f'Optional summary figure not found: {summary_figure}')

## 13. Final Model-Selection Checklist

- Confirm every final run used the non-overlapped PinMyMetal split, or clearly label exact/possibly-overlapped runs as secondary reference only.
- Compare Only-GVP, Only-ESM, GVP + late fusion, and GVP + early fusion before moving to complex fusion modes.
- Select checkpoints and model variants by validation metrics, not by repeatedly checking the held-out test set.
- For metal prediction, report both 6-class metrics and collapsed-4 metrics from the selected checkpoint.
- For EC prediction, report supported EC level metrics from the selected checkpoint.
- Preserve `run_config.json`, `run_metadata.json`, `dataset_summary.json`, `test_report.json`, selected checkpoint path, split identity, overlap status, and the summary CSV/figure in Drive.